# Echoes of Arcana — Google Colab Training (T4 GPU / CPU → `spell_model.onnx`)

В меню Colab выберите среду выполнения с GPU T4. Для пробного запуска оставьте `UPLOAD_CUSTOM_ZIP = False`.
Для обучения на записях команды загрузите архив из `pack_dataset.py` и установите `UPLOAD_CUSTOM_ZIP = True`.
При желании укажите `HOLDOUT_SPEAKER` до запуска. Все записи этого участника попадут в валидацию.

Итоговый `voice_training_artifacts.zip` содержит ONNX, `val_accuracy.png`, `confusion_matrix.png` и
`training_metrics.json`. Метрики стартового датасета описывают разделение по файлам. Метрики командного
датасета описывают один отложенный голос.

In [ ]:
# Ячейка 1: Проверка GPU и установка зависимостей (включая onnxscript)
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

!pip -q install onnx onnxscript onnxruntime scikit-learn matplotlib soundfile

In [ ]:
# Ячейка 2: Чтение одного архива с четырьмя классами
import hashlib
import shutil
import tempfile
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path, PurePosixPath

UPLOAD_CUSTOM_ZIP = False
CUSTOM_ZIP_PATH = ""  # опционально: путь к ZIP (например /content/dataset_hybrid_ali.zip)
HOLDOUT_SPEAKER = ""  # ID участника из имени WAV, например ali
RAW_REPO = "https://raw.githubusercontent.com/Alizhan2/Game-x"
LABELS = ["background", "virdan", "noktar", "targum"]
DATA_DIR = Path(tempfile.mkdtemp(prefix="echoes-voice-"))

if UPLOAD_CUSTOM_ZIP:
    candidate_zips = [Path(CUSTOM_ZIP_PATH)] if CUSTOM_ZIP_PATH.strip() else []
    candidate_zips.extend([Path("/content/dataset_hybrid_ali.zip"), Path("/content/dataset_raw.zip")])
    existing = next((p for p in candidate_zips if p.is_file()), None)
    if existing is not None:
        archive_path = DATA_DIR / existing.name
        shutil.copy2(existing, archive_path)
        print(f"Использую загруженный архив: {existing}")
    else:
        from google.colab import files

        uploaded = files.upload()
        if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith(".zip"):
            raise ValueError("Загрузите один ZIP, созданный pack_dataset.py.")
        archive_name, archive_bytes = next(iter(uploaded.items()))
        archive_path = DATA_DIR / Path(archive_name).name
        archive_path.write_bytes(archive_bytes)
else:
    archive_path = DATA_DIR / "bootstrap_dataset.zip"
    urllib.request.urlretrieve(f"{RAW_REPO}/main/ai_voice/bootstrap_dataset.zip", archive_path)

archive_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
with zipfile.ZipFile(archive_path) as archive:
    seen = set()
    for member in archive.infolist():
        parts = PurePosixPath(member.filename).parts
        if len(parts) != 3 or parts[0] != "dataset_raw" or parts[1] not in LABELS:
            continue
        filename = parts[2]
        if not filename.lower().endswith(".wav") or chr(92) in filename:
            continue
        key = (parts[1], filename.casefold())
        if key in seen:
            raise ValueError(f"Одинаковые имена WAV в архиве: {member.filename}")
        seen.add(key)
        output_path = DATA_DIR / parts[1] / filename
        output_path.parent.mkdir(exist_ok=True)
        with archive.open(member) as reader, output_path.open("wb") as writer:
            shutil.copyfileobj(reader, writer)

items = [
    (wav_path, idx)
    for idx, label in enumerate(LABELS)
    for wav_path in sorted((DATA_DIR / label).glob("*.wav"))
]
counts = Counter(label for _, label in items)
found = {label: counts[idx] for idx, label in enumerate(LABELS)}
print("Найдено клипов по классам:", found)
minimum = 10 if UPLOAD_CUSTOM_ZIP else 4
if any(counts[idx] < minimum for idx in range(len(LABELS))):
    raise ValueError(f"Нужно минимум {minimum} записей каждого класса. Найдено: {found}")

In [ ]:
# Ячейка 3: Отложенный диктор для записей команды. Только train подвергается аугментации
import random
import re
import sys
import numpy as np
import soundfile as sf

# Препроцессинг из зафиксированной версии движка, которая выпустила базовую ONNX-модель.
PREPROCESS_REF = "912a6aa3c384b84913619fcafefb3f098371df24"
if not (Path("ai_voice") / "features.py").is_file():
    urllib.request.urlretrieve(f"{RAW_REPO}/{PREPROCESS_REF}/ai_voice/features.py", DATA_DIR / "features.py")
    sys.path.insert(0, str(DATA_DIR))
else:
    sys.path.insert(0, str(Path("ai_voice").resolve()))
from features import (
    LABELS as ENGINE_LABELS, TARGET_SR, TARGET_SAMPLES,
    augment_wave, extract_log_mel, pad_or_trim, resample_to_16k,
)
assert LABELS == ENGINE_LABELS

seed = 42
rng = random.Random(seed)
if UPLOAD_CUSTOM_ZIP:
    by_speaker = {}
    for item in items:
        path, _ = item
        prefix, separator, recording = path.stem.partition("__")
        styled = re.fullmatch(r"([a-zA-Z0-9_-]+)_(?:normal|whisper|loud|shout|noise)_\d+(?:_\d{10,})?", recording)
        legacy = re.fullmatch(r"([a-zA-Z0-9_-]+?)(?:_\d{10,})?_\d+", recording)
        match = styled or legacy
        if not separator or not match or prefix.lower() != LABELS[item[1]]:
            raise ValueError(f"Нужен ID диктора и класс в имени WAV: {path.name}")
        by_speaker.setdefault(match.group(1).lower(), []).append(item)

    all_classes = set(range(len(LABELS)))
    full_speakers = [s for s, group in by_speaker.items() if {label for _, label in group} == all_classes]
    if len(full_speakers) >= 3 and len(full_speakers) == len(by_speaker):
        holdout = HOLDOUT_SPEAKER.strip().lower() or min(
            by_speaker, key=lambda speaker: (abs(len(by_speaker[speaker]) - len(items) / len(by_speaker)), speaker)
        )
        if holdout not in by_speaker:
            raise ValueError(f"Диктор {holdout!r} отсутствует. Доступны: {', '.join(sorted(by_speaker))}.")
        val_items = by_speaker[holdout]
        train_items = [item for speaker, group in by_speaker.items() if speaker != holdout for item in group]
        split_type = "speaker_holdout"
    else:
        train_items, val_items = [], []
        holdout = None
        split_type = "hybrid_stratified_files"
        for is_boot in (False, True):
            for label in range(len(LABELS)):
                group = [
                    item for item in items
                    if item[1] == label and (("__boot" in item[0].name or item[0].name.startswith("background__noise_")) == is_boot)
                ]
                if not group:
                    continue
                rng.shuffle(group)
                split_at = max(1, round(len(group) * 0.25))
                val_items.extend(group[:split_at])
                train_items.extend(group[split_at:])
else:
    train_items, val_items = [], []
    holdout = None
    split_type = "bootstrap_stratified_files"
    for label in range(len(LABELS)):
        group = [item for item in items if item[1] == label]
        rng.shuffle(group)
        split_at = max(1, round(len(group) * 0.25))
        val_items.extend(group[:split_at])
        train_items.extend(group[split_at:])

def read_clip(path):
    wave, sample_rate = sf.read(str(path), dtype="float32", always_2d=True)
    if not len(wave) or not np.isfinite(wave).all():
        raise ValueError(f"Пустой или повреждённый WAV: {path.name}")
    return pad_or_trim(resample_to_16k(wave.mean(axis=1), sample_rate), TARGET_SAMPLES)

x_tr, y_tr = [], []
for wav_path, idx in train_items:
    wave = read_clip(wav_path)
    x_tr.append(extract_log_mel(wave)[0])
    y_tr.append(idx)
    for _ in range(10):
        x_tr.append(extract_log_mel(augment_wave(wave, rng))[0])
        y_tr.append(idx)

X_tr = np.stack(x_tr)
Y_tr = np.asarray(y_tr, dtype=np.int64)
X_val = np.stack([extract_log_mel(read_clip(path))[0] for path, _ in val_items])
Y_val = np.asarray([label for _, label in val_items], dtype=np.int64)
print(f"Train: {len(train_items)} original clips | Validation: {len(val_items)} original clips")
print(f"Split: {split_type} | Held-out speaker: {holdout or 'n/a'}")

In [ ]:
# Ячейка 4: Mini-batch обучение SpellCNN + сохранение метрик и графиков для жюри
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

class SpellCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 48, kernel_size=3, padding=1),
            nn.BatchNorm2d(48),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.dropout = nn.Dropout(0.15)
        self.classifier = nn.Linear(48, num_classes)

    def forward(self, x):
        feat = self.dropout(self.features(x).flatten(1))
        return self.classifier(feat)

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SpellCNN(len(LABELS)).to(device)
EPOCHS = 45
opt = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss()

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(Y_tr)),
    batch_size=32,
    shuffle=True,
)
X_val_t, Y_val_t = torch.from_numpy(X_val).to(device), torch.from_numpy(Y_val).to(device)

history_loss, history_acc = [], []
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        opt.zero_grad()
        loss = loss_fn(model(bx), by)
        loss.backward()
        opt.step()
        total_loss += float(loss.item()) * len(bx)
    scheduler.step()
    avg_loss = total_loss / max(1, len(X_tr))

    model.eval()
    with torch.no_grad():
        preds = model(X_val_t).argmax(dim=1)
        acc = float((preds == Y_val_t).float().mean()) * 100.0
    history_loss.append(avg_loss)
    history_acc.append(acc)
    if epoch % 5 == 0 or epoch == EPOCHS:
        print(f"Epoch {epoch:02d}/{EPOCHS} | loss={avg_loss:.4f} | val_acc={acc:.1f}%")

model.eval()
with torch.no_grad():
    val_preds = model(X_val_t).argmax(dim=1).cpu().numpy()

live_mask = np.array([not ("__boot" in path.name or path.name.startswith("background__noise_")) for path, _ in val_items])
live_val_acc = float((val_preds[live_mask] == Y_val[live_mask]).mean() * 100.0) if live_mask.any() else None

metrics = {
    "dataset": "team_recordings" if UPLOAD_CUSTOM_ZIP else "bootstrap",
    "dataset_sha256": archive_sha256,
    "split": split_type,
    "held_out_speaker": holdout,
    "seed": seed,
    "device": torch.cuda.get_device_name(0) if device == "cuda" else "CPU",
    "epochs": EPOCHS,
    "final_val_acc_pct": round(history_acc[-1], 2),
    "live_val_acc_pct": round(live_val_acc, 2) if live_val_acc is not None else None,
    "live_val_clips": int(live_mask.sum()),
    "final_train_loss": round(history_loss[-1], 4),
    "train_original_clips": len(train_items),
    "train_samples_with_augmentation": len(X_tr),
    "validation_original_clips": len(X_val),
    "validation_clips_by_class": {label: int((Y_val == i).sum()) for i, label in enumerate(LABELS)},
    "labels": LABELS,
    "val_accuracy_by_epoch_pct": [round(acc, 2) for acc in history_acc],
}
ARTIFACTS = Path("voice_training_artifacts")
ARTIFACTS.mkdir(exist_ok=True)
(ARTIFACTS / "training_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"FINAL_VAL_ACC={metrics['final_val_acc_pct']}%")
if live_val_acc is not None:
    print(f"LIVE_VAL_ACC={metrics['live_val_acc_pct']}% (n={metrics['live_val_clips']})")
print(classification_report(Y_val, val_preds, labels=range(len(LABELS)), target_names=LABELS, zero_division=0))

fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.plot(range(1, EPOCHS + 1), history_acc, color="#0369a1")
ax.set(title=f"Validation accuracy ({split_type})", xlabel="Epoch", ylabel="Accuracy (%)", ylim=(0, 100))
ax.grid(True, alpha=0.3)
fig.savefig(ARTIFACTS / "val_accuracy.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
cm = confusion_matrix(Y_val, val_preds, labels=range(len(LABELS)))
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title("Confusion matrix · original validation clips")
fig.savefig(ARTIFACTS / "confusion_matrix.png", dpi=160)
plt.show()

In [ ]:
# Ячейка 5: Проверка совпадения логитов PyTorch и ONNX Runtime на всей валидации
import inspect
import onnxruntime as ort

model_cpu = model.eval().cpu()
sample = torch.from_numpy(X_val[:1])
model_path = ARTIFACTS / "spell_model.onnx"
export_kwargs = {
    "input_names": ["mel"],
    "output_names": ["logits"],
    "dynamic_axes": {"mel": {0: "batch"}, "logits": {0: "batch"}},
    "opset_version": 17,
}
if "dynamo" in inspect.signature(torch.onnx.export).parameters:
    export_kwargs["dynamo"] = False
torch.onnx.export(model_cpu, sample, str(model_path), **export_kwargs)

session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
np_logits = session.run(["logits"], {"mel": X_val})[0]
with torch.no_grad():
    torch_logits = model_cpu(torch.from_numpy(X_val)).numpy()

np.testing.assert_allclose(np_logits, torch_logits, rtol=1e-4, atol=1e-4)
metrics["onnx_max_abs_logit_error"] = float(np.max(np.abs(np_logits - torch_logits)))
metrics["onnx_sha256"] = hashlib.sha256(model_path.read_bytes()).hexdigest()
(ARTIFACTS / "training_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

artifact_names = ("spell_model.onnx", "val_accuracy.png", "confusion_matrix.png", "training_metrics.json")
archive_file = Path("voice_training_artifacts.zip")
with zipfile.ZipFile(archive_file, "w", compression=zipfile.ZIP_DEFLATED) as output:
    for name in artifact_names:
        output.write(ARTIFACTS / name, arcname=name)
print(f"ONNX проверен; артефакты сохранены в {archive_file}")

if "google.colab" in sys.modules:
    from google.colab import files

    try:
        files.download(str(archive_file))
    except Exception as exc:
        print("Скачайте вручную из панели Файлы:", archive_file, exc)